# 📖 Notebook 1: Partition Keys and Sort Keys

The **primary key** is the most important decision you make when designing a DynamoDB table. It determines how your data is stored, distributed, and queried.

## Learning Objectives

By the end of this notebook, you'll understand:
- How DynamoDB organizes data into tables, items, and attributes
- What a partition key is and why it matters
- What a sort key is and when to use one
- How to perform basic CRUD operations
- How to use Query vs Scan and why Query is preferred

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 03-technologies/databases/dynamodb
docker compose up -d
```

### Visualization Tools

- **DynamoDB Admin GUI**: http://localhost:8001  
  Browse tables and items as you work through the notebook.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import boto3
from botocore.exceptions import ClientError
import json
import time

# Connect to DynamoDB Local (no real AWS credentials needed)
dynamodb = boto3.resource(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

# Also create a low-level client (useful for some operations)
client = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

# Test the connection
try:
    tables = client.list_tables()["TableNames"]
    print("✅ Connected to DynamoDB Local")
    print(f"   Existing tables: {tables if tables else '(none)'}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker compose up -d")

## 🧱 DynamoDB Data Model

DynamoDB organizes data into three levels:

| Concept | SQL Equivalent | Description |
|---------|---------------|-------------|
| **Table** | Table | Top-level container for data |
| **Item** | Row | A single record (up to 400KB) |
| **Attribute** | Column | A key-value pair within an item |

**Key difference from SQL**: DynamoDB is **schema-less**. Items in the same table can have different attributes. You only define the primary key attributes up front — everything else is flexible.

Think of it like a filing cabinet:
- The **table** is the cabinet
- Each **item** is a folder in the cabinet
- Each **attribute** is a piece of paper in the folder
- Some folders may have more papers than others — that's perfectly fine!

## 🔑 Part 1: Partition Key Only (Simple Primary Key)

The simplest table has just a **partition key** — a single attribute that uniquely identifies each item.

DynamoDB **hashes** the partition key to decide which physical storage partition the item lives on. This is how DynamoDB distributes data across many servers.

Think of it like a post office: the partition key is the ZIP code, and DynamoDB uses it to route your letter to the right sorting facility.

### Rules for partition keys:
- Must be **unique** across all items (when there's no sort key)
- Should distribute data **evenly** across partitions
- Good choices: user_id, product_id, email
- Bad choices: country (too few values), status ("active"/"inactive")

In [ ]:
# Create a simple Users table with just a partition key

TABLE_NAME = "Users"

# Clean up if the table already exists from a previous run
try:
    dynamodb.Table(TABLE_NAME).delete()
    dynamodb.Table(TABLE_NAME).wait_until_not_exists()
    print(f"🗑️  Deleted existing '{TABLE_NAME}' table")
except ClientError:
    pass

# Create the table
table = dynamodb.create_table(
    TableName=TABLE_NAME,
    # Only define key attributes in the schema — everything else is flexible
    KeySchema=[
        {"AttributeName": "user_id", "KeyType": "HASH"},  # HASH = partition key
    ],
    AttributeDefinitions=[
        {"AttributeName": "user_id", "AttributeType": "S"},  # S = String
    ],
    BillingMode="PAY_PER_REQUEST",  # On-demand pricing (no capacity planning)
)

# Wait for the table to be ready
table.wait_until_exists()
print(f"✅ Created table '{TABLE_NAME}'")
print(f"   Key schema: partition key = user_id (String)")
print(f"   Status: {table.table_status}")

In [ ]:
# Insert some users — notice each item can have DIFFERENT attributes!
# This is the schema-less nature of DynamoDB.

users = [
    {
        "user_id": "user-101",
        "name": "Alice Smith",
        "email": "alice@example.com",
        "phone": "555-1234",
    },
    {
        "user_id": "user-102",
        "name": "Bob Jones",
        "email": "bob@example.com",
        "address": {  # Nested attribute — DynamoDB supports complex types
            "street": "123 Main St",
            "city": "Seattle",
            "state": "WA",
        },
    },
    {
        "user_id": "user-103",
        "name": "Carol Williams",
        "email": "carol@example.com",
        "favorite_color": "blue",  # Only Carol has this attribute
    },
]

for user in users:
    table.put_item(Item=user)
    print(f"✅ Inserted {user['name']} (attributes: {list(user.keys())})")

print()
print("💡 Notice: Alice has 'phone', Bob has 'address', Carol has 'favorite_color'.")
print("   In SQL, every row must have the same columns. In DynamoDB, items are flexible!")

In [ ]:
# Read a single item by its partition key — this is called GetItem
# GetItem is the fastest way to read from DynamoDB (single-digit millisecond)

response = table.get_item(Key={"user_id": "user-102"})
item = response.get("Item")

print("📦 GetItem result for user-102:")
print(json.dumps(item, indent=2, default=str))
print()
print("💡 GetItem requires the FULL primary key. It's O(1) — always fast, regardless of table size.")

In [ ]:
# Update an item — add a new attribute to Alice

table.update_item(
    Key={"user_id": "user-101"},
    UpdateExpression="SET age = :a, #s = :s",
    ExpressionAttributeNames={"#s": "status"},  # 'status' is a reserved word
    ExpressionAttributeValues={":a": 30, ":s": "active"},
)

# Read it back to verify
updated = table.get_item(Key={"user_id": "user-101"})["Item"]
print("📝 Updated Alice:")
print(json.dumps(updated, indent=2, default=str))
print()
print("💡 We added 'age' and 'status' to Alice without changing the table schema.")
print("   Bob and Carol are NOT affected — schema-less!")

In [ ]:
# Delete an item by its partition key

table.delete_item(Key={"user_id": "user-103"})
print("🗑️  Deleted user-103 (Carol)")

# Verify it's gone
response = table.get_item(Key={"user_id": "user-103"})
print(f"   GetItem result: {response.get('Item', 'Not found')}")

## 🔑🔑 Part 2: Partition Key + Sort Key (Composite Primary Key)

Many real-world use cases need to store **multiple related items** that you want to query together. This is where the **sort key** comes in.

With a composite key:
- The **partition key** groups related items together (on the same physical partition)
- The **sort key** orders items within that group (stored in a B-tree)
- Together, they **uniquely identify** each item

Think of it like a bookshelf:
- The **partition key** is the shelf (all books by one author go on the same shelf)
- The **sort key** is the position on the shelf (books ordered by publish date)

### Example: Chat Messages
A group chat has many messages. We want to:
1. Get all messages for a chat (partition key = `chat_id`)
2. Get them in chronological order (sort key = `message_id`)

In [ ]:
# Create a ChatMessages table with a composite primary key

CHAT_TABLE = "ChatMessages"

try:
    dynamodb.Table(CHAT_TABLE).delete()
    dynamodb.Table(CHAT_TABLE).wait_until_not_exists()
    print(f"🗑️  Deleted existing '{CHAT_TABLE}' table")
except ClientError:
    pass

chat_table = dynamodb.create_table(
    TableName=CHAT_TABLE,
    KeySchema=[
        {"AttributeName": "chat_id", "KeyType": "HASH"},   # Partition key
        {"AttributeName": "message_id", "KeyType": "RANGE"}, # Sort key (RANGE)
    ],
    AttributeDefinitions=[
        {"AttributeName": "chat_id", "AttributeType": "S"},
        {"AttributeName": "message_id", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",
)

chat_table.wait_until_exists()
print(f"✅ Created table '{CHAT_TABLE}'")
print(f"   Partition key: chat_id (String)")
print(f"   Sort key: message_id (String)")

In [ ]:
# Insert messages into two different chat groups

messages = [
    # Chat group: general
    {"chat_id": "general", "message_id": "msg-001", "sender": "Alice", "text": "Hey everyone!", "timestamp": "2024-01-15T10:00:00Z"},
    {"chat_id": "general", "message_id": "msg-002", "sender": "Bob",   "text": "Hi Alice!",     "timestamp": "2024-01-15T10:01:00Z"},
    {"chat_id": "general", "message_id": "msg-003", "sender": "Carol", "text": "Good morning!", "timestamp": "2024-01-15T10:02:00Z"},
    {"chat_id": "general", "message_id": "msg-004", "sender": "Alice", "text": "How is everyone?", "timestamp": "2024-01-15T10:05:00Z"},
    {"chat_id": "general", "message_id": "msg-005", "sender": "Bob",   "text": "Doing great!",  "timestamp": "2024-01-15T10:06:00Z"},
    # Chat group: engineering
    {"chat_id": "engineering", "message_id": "msg-001", "sender": "Dave",  "text": "Deploy is done",   "timestamp": "2024-01-15T11:00:00Z"},
    {"chat_id": "engineering", "message_id": "msg-002", "sender": "Alice", "text": "Great, thanks!",   "timestamp": "2024-01-15T11:01:00Z"},
    {"chat_id": "engineering", "message_id": "msg-003", "sender": "Dave",  "text": "No issues found", "timestamp": "2024-01-15T11:05:00Z"},
]

with chat_table.batch_writer() as batch:
    for msg in messages:
        batch.put_item(Item=msg)

print(f"✅ Inserted {len(messages)} messages across 2 chat groups")
print()
print("💡 Notice: message_id 'msg-001' appears in BOTH chats.")
print("   That's fine! The primary key is (chat_id + message_id) combined.")

In [ ]:
# QUERY: Get all messages for the 'general' chat, sorted by message_id
# Query is EFFICIENT — it only reads items that match the partition key.

from boto3.dynamodb.conditions import Key

response = chat_table.query(
    KeyConditionExpression=Key("chat_id").eq("general")
)

print("📨 Query: All messages in 'general' chat")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['message_id']}] {item['sender']}: {item['text']}")

print(f"\n📊 Items returned: {response['Count']}")
print(f"   Data scanned: {response['ScannedCount']} items")
print()
print("💡 Query reads ONLY the 'general' partition — it never touches 'engineering'.")
print("   Items come back sorted by sort key (message_id) automatically.")

In [ ]:
# QUERY with sort key conditions — get only recent messages
# The sort key supports range operations: between, >, <, begins_with

response = chat_table.query(
    KeyConditionExpression=(
        Key("chat_id").eq("general") &
        Key("message_id").between("msg-003", "msg-005")
    )
)

print("📨 Query: Messages msg-003 to msg-005 in 'general'")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['message_id']}] {item['sender']}: {item['text']}")

print(f"\n💡 Sort key conditions let you do range queries WITHIN a partition.")
print("   This is powered by a B-tree index under the hood.")

In [ ]:
# QUERY in reverse order — newest messages first

response = chat_table.query(
    KeyConditionExpression=Key("chat_id").eq("general"),
    ScanIndexForward=False,  # Reverse sort order
    Limit=3,                 # Only get the 3 most recent
)

print("📨 Query: Last 3 messages in 'general' (newest first)")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['message_id']}] {item['sender']}: {item['text']}")

print(f"\n💡 ScanIndexForward=False reverses the sort key order.")
print("   Combined with Limit, this is perfect for 'show latest N' features.")

## 🔍 Part 3: Query vs Scan

DynamoDB gives you two ways to read multiple items:

| Operation | What It Does | Performance | Cost |
|-----------|-------------|-------------|------|
| **Query** | Reads items matching a partition key | ⚡ Fast | 💰 Low |
| **Scan** | Reads EVERY item in the table | 🐌 Slow | 💸 High |

**Rule of thumb**: Always prefer Query. Use Scan only for admin tasks or small tables.

Let's see the difference.

In [ ]:
# SCAN: Read ALL items in the table (expensive!)

response = chat_table.scan()

print("📋 Scan: ALL messages across ALL chats")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['chat_id']}] [{item['message_id']}] {item['sender']}: {item['text']}")

print(f"\n📊 Scanned count: {response['ScannedCount']} items")
print()
print("⚠️  Scan reads EVERY item in the table, even if you only need a few.")
print("   With 1 million items, Scan reads all 1 million. Query reads only your partition.")
print()
print("📏 Cost comparison for a table with 1M items, 100 items per chat:")
print("   Query 'general' chat: reads ~100 items   → fast, cheap")
print("   Scan entire table:    reads 1,000,000 items → slow, expensive")

## 🧪 Part 4: Choosing Good Partition Keys

Your partition key directly affects performance. A good key **spreads data evenly** across partitions. A bad key creates **hot partitions** — one partition gets all the traffic while others sit idle.

| ✅ Good Partition Keys | ❌ Bad Partition Keys |
|----------------------|---------------------|
| user_id (high cardinality) | country (too few values) |
| order_id (unique per order) | status ("active"/"inactive") |
| device_id (one per device) | date (all today's writes hit one partition) |
| chat_id (one per conversation) | is_premium (true/false) |

In [ ]:
# Let's visualize what happens with good vs bad partition keys

import random

def simulate_partition_distribution(key_func, num_items=1000, num_partitions=8):
    """Simulate how items distribute across partitions."""
    partitions = [0] * num_partitions
    for i in range(num_items):
        key = key_func(i)
        partition = hash(key) % num_partitions
        partitions[partition] += 1
    return partitions

# Good key: unique user_id
good_dist = simulate_partition_distribution(
    key_func=lambda i: f"user-{i}",
)

# Bad key: only 3 possible values
bad_dist = simulate_partition_distribution(
    key_func=lambda i: random.choice(["active", "inactive", "pending"]),
)

print("📊 Partition Distribution (1000 items across 8 partitions)")
print("=" * 60)
print()
print("✅ Good key (user_id — high cardinality):")
for i, count in enumerate(good_dist):
    bar = "█" * (count // 5)
    print(f"   Partition {i}: {bar} ({count})")

print()
print("❌ Bad key (status — only 3 values):")
for i, count in enumerate(bad_dist):
    bar = "█" * (count // 5)
    print(f"   Partition {i}: {bar} ({count})")

print()
print("💡 The good key spreads items evenly. The bad key creates hot partitions.")
print("   Hot partitions can cause throttling — DynamoDB limits each partition to")
print("   3,000 RCU and 1,000 WCU.")

## 🎯 Key Takeaways

1. **Partition key** determines WHERE data is stored (which physical partition)
2. **Sort key** determines HOW data is ordered within a partition
3. **Together** they uniquely identify each item
4. **Query** is efficient (reads one partition); **Scan** is expensive (reads everything)
5. Choose partition keys with **high cardinality** to avoid hot partitions
6. Use sort keys to enable **range queries** and **sorting** within a partition

### Under the Hood
- Partition key → **hashed** to determine the storage node
- Sort key → stored in a **B-tree** within each partition for efficient range queries
- Query first finds the partition (via hash), then traverses the B-tree (via sort key)

### Next Up
In the next notebook, we'll learn about **Secondary Indexes** — how to query data by attributes that are NOT part of the primary key.